# Unidade 3 - Bloco prático da Aula 01: TPE vs Random no Optuna

Compara o RandomSampler e o TPESampler do Optuna com o mesmo orçamento de 40 tentativas sobre um gradient boosting. Além do melhor resultado único, olhe a média das últimas tentativas de cada método: é ali que a estratégia do TPE aparece.

In [ ]:
import numpy as np
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objetivo(trial):
    modelo = GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3,
                                          log=True),
        max_depth=trial.suggest_int("max_depth", 2, 6),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        random_state=42)
    return cross_val_score(modelo, X, y, cv=cv, scoring="f1",
                           n_jobs=-1).mean()

estudos = {}
for nome, sampler in [("Random", optuna.samplers.RandomSampler(seed=42)),
                      ("TPE", optuna.samplers.TPESampler(seed=42))]:
    estudo = optuna.create_study(direction="maximize", sampler=sampler)
    estudo.optimize(objetivo, n_trials=40)
    estudos[nome] = estudo
    vals = [t.value for t in estudo.trials]
    marcos = {n: max(vals[:n]) for n in (10, 20, 40)}
    print(f"{nome:>6}: marcos {marcos} | media das 10 ultimas: "
          f"{np.mean(vals[-10:]):.4f}")

imp = optuna.importance.get_param_importances(estudos["TPE"])
print("importancia de cada hiperparametro:",
      {k: round(v, 2) for k, v in imp.items()})